# Vanishing-Act — a quantitative teardown 🔬
### SMB with a Lo t-stat · small-vs-large Sharpe · the sign reversal since 2010 · quality-not-size

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Survived publication?: Busted](https://img.shields.io/badge/Survived_publication%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We test the size premium on three pairs and find it statistically zero and decaying to negative.

> ⚠️ **Not investment advice.** ^RUT/^GSPC (1987–2026) + IWM/SPY + IJR/IVV total return (2000–2026), Yahoo. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (vanishing_act/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from vanishing_act import data, strategy as st
ret = data.fetch_pairs()                       # cache-first; built by examples/verify.py --fetch
smb_long = st.smb(ret, "^RUT", "^GSPC")        # the long-history spread


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | 39-yr SMB Sharpe 0.01, Lo t 0.05 |
| Tradability | **Mirage** | ETF SMB insignificant; small Sharpe < large |
| Survived publication? | **Busted** | positive pre-2010, negative since, every pair |

> 💡 *In plain words:* the original factor is the clearest decay case on the bench.

## 1 · The claim, steelmanned

- **H₁:** SMB > 0 with |t| > 2 over the long sample.
- **H₂:** small-caps deliver a higher Sharpe than large-caps.
- **H₃:** the premium is stable (didn't decay after Banz 1981).

## 2 · So what? — what rides on each

If H₁/H₂ hold, a size tilt is a free risk-adjusted gain. If they fail, the SMB factor and small-cap allocations rest on a dead premium.

## 3 · How we'd know — the protocol

SMB spread per pair → Lo (2002) t-stat → each leg's standalone Sharpe → pre/post-2010 decay split → the quality caveat (S&P 600 vs Russell 2000).

## 4 · The teardown

### 4.1 The spread, with a t-stat, across pairs

In [2]:
rows={}
for k,(a,b) in {'^RUT−^GSPC':('^RUT','^GSPC'),'IWM−SPY':('IWM','SPY'),'IJR−IVV':('IJR','IVV')}.items():
    if a in ret and b in ret:
        s=st.smb_stats(st.smb(ret,a,b)); rows[k]={'SMB %/yr':s['mean_ann']*100,'Sharpe':s['sharpe'],'Lo t':s['tstat'],'n':s['n']}
display(pd.DataFrame(rows).T.round(3))

,SMB %/yr,Sharpe,Lo t,n
^RUT−^GSPC,0.084,0.008,0.047,465.0
IWM−SPY,0.988,0.098,0.500,313.0
IJR−IVV,2.110,0.205,1.047,313.0


> 💡 *In plain words:* not one |t| reaches 2. **H₁ rejected** — no detectable premium.

### 4.2 Small-caps trail on Sharpe

In [3]:
for a,b in [('^RUT','^GSPC'),('IWM','SPY'),('IJR','IVV')]:
    if a in ret and b in ret:
        print(f'{a} Sharpe {st.leg_summary(ret,a)["sharpe"]:.2f}  vs  {b} Sharpe {st.leg_summary(ret,b)["sharpe"]:.2f}')

^RUT Sharpe 0.47  vs  ^GSPC Sharpe 0.67
IWM Sharpe 0.51  vs  SPY Sharpe 0.76
IJR Sharpe 0.58  vs  IVV Sharpe 0.60


> 💡 *In plain words:* the 'risk premium' delivers *less* risk-adjusted return than the large-cap index. **H₂ rejected.**

### 4.3 The decay — positive then negative

In [4]:
pairs={'^RUT−^GSPC':('^RUT','^GSPC'),'IWM−SPY':('IWM','SPY'),'IJR−IVV':('IJR','IVV')}
dec=pd.concat({k: st.decay_split(st.smb(ret,a,b))['mean_ann'] for k,(a,b) in pairs.items() if a in ret and b in ret}, axis=1).T
display((dec*100).round(2))

,pre-2010,2010-on
^RUT−^GSPC,1.11,-1.30
IWM−SPY,5.75,-1.78
IJR−IVV,7.69,-1.13


> 💡 *In plain words:* every pair's premium is positive before 2010 and negative after. **H₃ rejected** — the cleanest post-publication reversal on the bench (van Dijk 2011).

### 4.4 What survives is quality, not size

The one pair with a faint positive tilt is **IJR − IVV** (S&P SmallCap 600 vs S&P 500). The S&P 600, unlike the Russell 2000, **screens constituents for positive earnings** — so its edge is a *profitability* screen, exactly the Asness et al. (2018) *"Size Matters, If You Control Your Junk"* result. Strip the quality and the size premium is gone.

## 5 · The verdict

H₁, H₂, H₃ all rejected → Signal `NONE`, Tradability `MIRAGE`, survived-publication `BUSTED`.

## 6 · Could you trade it?

A size *tilt* buys a lower Sharpe than the S&P 500. The only live version is a quality screen in disguise. There is no size premium to harvest on tradable proxies.

## 7 · Going further

Forks: (a) Ken French's SMB back to 1926 — confirm the pre-1981 premium and the post-1981 fade on the academic series; (b) double-sort size × profitability to isolate the quality driver; (c) micro-cap SMB, where survivorship and illiquidity inflate the headline. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).